# S6E2 Ensemble
Combine CatBoost, LightGBM, XGBoost (+ optionally LR) using OOF-based weight optimization.

## Strategy
1. Generate **out-of-fold (OOF) probabilities** for each model — critical to avoid overfitting weights
2. **Monte Carlo** simplex sampling (10k samples) with 3D Plotly visualization
3. **SLSQP** constrained optimization for comparison (same answer, much faster)
4. Test 3-model (GBTs only) and 4-model (GBTs + LR) ensembles
5. Submit best

## Model params — update after tuning results
Swap in tuned params below once `s6e2-lgbm-xgb-tuning.ipynb` finishes.

## Imports & Data

In [14]:
import numpy as np
import pandas as pd
import subprocess
import random
from pathlib import Path
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
import plotly.express as px
import plotly.graph_objects as go

KAGGLE_DATA = Path("/kaggle/input/playground-series-s6e2")
LOCAL_DATA  = Path("data")
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    if "heart_disease" in df.columns:
        df["heart_disease"] = df["heart_disease"].map({"Absence": 0, "Presence": 1})
    return df

train = prep(pd.read_csv(DATA_DIR / "train.csv"))
test  = prep(pd.read_csv(DATA_DIR / "test.csv"))
ss    = pd.read_csv(DATA_DIR / "sample_submission.csv")

FEATURES     = [c for c in train.columns if c not in ["heart_disease", "id"]]
CAT_FEATURES = ["sex", "chest_pain_type", "fbs_over_120", "ekg_results",
                "exercise_angina", "slope_of_st", "number_of_vessels_fluro", "thallium"]

X = train[FEATURES]; y = train["heart_disease"]; X_test = test[FEATURES]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X.shape}  Test: {X_test.shape}")

Train: (630000, 13)  Test: (270000, 13)


## Model Params
Update with tuned values from `s6e2-lgbm-xgb-tuning.ipynb` once available.

In [15]:
# ── Tuned params (ES lr=0.05, grid showed no further gain) ──────────────
CAT_PARAMS = dict(
    iterations=500, learning_rate=0.1, depth=6,
    task_type="GPU", cat_features=CAT_FEATURES, random_state=42, verbose=0
)
LGB_PARAMS = dict(
    n_estimators=589, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    device="gpu", random_state=42, n_jobs=-1, verbose=-1
)
XGB_PARAMS = dict(
    n_estimators=461, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", device="cuda", random_state=42, n_jobs=-1, verbosity=0
)
# ─────────────────────────────────────────────────────────────────────────
print("CatBoost:", {k:v for k,v in CAT_PARAMS.items() if k != 'cat_features'})
print("LightGBM:", LGB_PARAMS)
print("XGBoost: ", XGB_PARAMS)

CatBoost: {'iterations': 500, 'learning_rate': 0.1, 'depth': 6, 'task_type': 'GPU', 'random_state': 42, 'verbose': 0}
LightGBM: {'n_estimators': 589, 'learning_rate': 0.05, 'num_leaves': 31, 'subsample': 0.8, 'colsample_bytree': 0.8, 'device': 'gpu', 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
XGBoost:  {'n_estimators': 461, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'eval_metric': 'logloss', 'device': 'cuda', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}


## Generate OOF Predictions
Train each model on K-1 folds, predict the held-out fold and the test set.
OOF predictions cover the full training set without leakage — safe to use for weight optimization.

In [16]:
def get_oof_preds(model_fn, X, y, X_test, cv):
    """Return (oof_proba, test_proba_mean_across_folds)."""
    oof  = np.zeros(len(X))
    test_preds = np.zeros((len(X_test), cv.get_n_splits()))
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = model_fn()
        m.fit(X_tr, y_tr)
        oof[val_idx]       = m.predict_proba(X_val)[:, 1]
        test_preds[:, fold] = m.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_val, oof[val_idx])
        print(f"  fold {fold+1}: auc={auc:.4f}")
    print(f"  OOF AUC: {roc_auc_score(y, oof):.4f}")
    return oof, test_preds.mean(axis=1)

print("=== CatBoost ===")
oof_cat, test_cat = get_oof_preds(
    lambda: cb.CatBoostClassifier(**CAT_PARAMS), X, y, X_test, cv)

print("\n=== LightGBM ===")
oof_lgb, test_lgb = get_oof_preds(
    lambda: lgb.LGBMClassifier(**LGB_PARAMS), X, y, X_test, cv)

print("\n=== XGBoost ===")
oof_xgb, test_xgb = get_oof_preds(
    lambda: xgb.XGBClassifier(**XGB_PARAMS), X, y, X_test, cv)

print("\n=== Logistic Regression ===")
# LR needs scaling; log1p on skewed features
def lr_model_fn():
    from sklearn.pipeline import Pipeline
    import numpy as np
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=42, n_jobs=-1))
    ])

# One-hot encode categoricals for LR
X_lr      = pd.get_dummies(X,      columns=CAT_FEATURES)
X_test_lr = pd.get_dummies(X_test, columns=CAT_FEATURES)
X_test_lr = X_test_lr.reindex(columns=X_lr.columns, fill_value=0)

oof_lr, test_lr = get_oof_preds(
    lr_model_fn, X_lr, y, X_test_lr, cv)


=== CatBoost ===
  fold 1: auc=0.9556
  fold 2: auc=0.9547
  fold 3: auc=0.9555
  fold 4: auc=0.9550
  fold 5: auc=0.9558
  OOF AUC: 0.9553

=== LightGBM ===
  fold 1: auc=0.9556
  fold 2: auc=0.9546
  fold 3: auc=0.9554
  fold 4: auc=0.9549
  fold 5: auc=0.9557
  OOF AUC: 0.9552

=== XGBoost ===
  fold 1: auc=0.9556
  fold 2: auc=0.9546
  fold 3: auc=0.9554
  fold 4: auc=0.9549
  fold 5: auc=0.9558
  OOF AUC: 0.9552

=== Logistic Regression ===
  fold 1: auc=0.9532
  fold 2: auc=0.9523
  fold 3: auc=0.9530
  fold 4: auc=0.9526
  fold 5: auc=0.9533
  OOF AUC: 0.9529


In [17]:
# Save OOF & test arrays for TabNet notebook
import numpy as np
np.save("submissions/oof_cat.npy",  oof_cat)
np.save("submissions/oof_lgb.npy",  oof_lgb)
np.save("submissions/oof_xgb.npy",  oof_xgb)
np.save("submissions/test_cat.npy", test_cat)
np.save("submissions/test_lgb.npy", test_lgb)
np.save("submissions/test_xgb.npy", test_xgb)
print("Saved — safe to stop here")

Saved — safe to stop here


## Simple Average Baseline
Hard to beat — check this first.

In [7]:
avg3 = (oof_cat + oof_lgb + oof_xgb) / 3
avg4 = (oof_cat + oof_lgb + oof_xgb + oof_lr) / 4
print(f"Simple avg (3 GBTs):      OOF AUC = {roc_auc_score(y, avg3):.5f}")
print(f"Simple avg (3 GBTs + LR): OOF AUC = {roc_auc_score(y, avg4):.5f}")
print(f"")
print(f"Individual OOF AUC:")
for name, oof in [("CatBoost", oof_cat), ("LightGBM", oof_lgb),
                  ("XGBoost",  oof_xgb), ("LR",       oof_lr)]:
    print(f"  {name:<12} {roc_auc_score(y, oof):.5f}")

Simple avg (3 GBTs):      OOF AUC = 0.95537
Simple avg (3 GBTs + LR): OOF AUC = 0.95517

Individual OOF AUC:
  CatBoost     0.95533
  LightGBM     0.95523
  XGBoost      0.95525
  LR           0.95288


## Monte Carlo Weight Search — 3 Models
Sample 10,000 random weight combinations from the simplex, score each on OOF AUC.
Visualize as an interactive 3D scatter colored by score.

In [8]:
N_SAMPLES = 10_000
random.seed(42)

mc3 = []
for _ in range(N_SAMPLES):
    a, b, c = random.random(), random.random(), random.random()
    total = a + b + c
    a /= total; b /= total; c /= total
    score = roc_auc_score(y, a*oof_cat + b*oof_lgb + c*oof_xgb)
    mc3.append([a, b, c, score])

mc3_df = pd.DataFrame(mc3, columns=["catboost", "lgbm", "xgb", "score"])
best3  = mc3_df.loc[mc3_df["score"].idxmax()]  # MAX for AUC

print(f"Best MC weights (3-model):")
print(f"  catboost={best3['catboost']:.3f}  lgbm={best3['lgbm']:.3f}  xgb={best3['xgb']:.3f}")
print(f"  OOF AUC = {best3['score']:.5f}")
print(f"  vs simple avg = {roc_auc_score(y, avg3):.5f}")

fig = px.scatter_3d(
    mc3_df, x="catboost", y="lgbm", z="xgb", color="score",
    color_continuous_scale="RdYlGn",
    title="Monte Carlo: OOF AUC vs ensemble weights (3-model)",
    labels={"score": "OOF AUC"}
)
fig.update_traces(marker=dict(size=2, opacity=0.6))
# Mark the best point
fig.add_trace(go.Scatter3d(
    x=[best3["catboost"]], y=[best3["lgbm"]], z=[best3["xgb"]],
    mode="markers", marker=dict(size=8, color="red", symbol="diamond"),
    name=f"best ({best3['score']:.4f})"
))
fig.show()

Best MC weights (3-model):
  catboost=0.548  lgbm=0.219  xgb=0.233
  OOF AUC = 0.95538
  vs simple avg = 0.95537


## SLSQP Optimization — 3 Models
Constrained gradient optimization: weights ≥ 0, sum = 1.
Finds the true optimum in ~50 evaluations.

In [9]:
oofs3 = [oof_cat, oof_lgb, oof_xgb]
names3 = ["catboost", "lgbm", "xgb"]

def neg_auc(weights, oofs):
    pred = sum(w * o for w, o in zip(weights, oofs))
    return -roc_auc_score(y, pred)

result3 = minimize(
    neg_auc, x0=[1/3, 1/3, 1/3], args=(oofs3,),
    method="SLSQP",
    bounds=[(0, 1)] * 3,
    constraints={"type": "eq", "fun": lambda w: w.sum() - 1}
)
opt3_weights = result3.x
opt3_auc     = -result3.fun

print("SLSQP optimal weights (3-model):")
for n, w in zip(names3, opt3_weights):
    print(f"  {n:<12} {w:.4f}")
print(f"  OOF AUC = {opt3_auc:.5f}")

SLSQP optimal weights (3-model):
  catboost     0.3333
  lgbm         0.3333
  xgb          0.3333
  OOF AUC = 0.95537


## Monte Carlo Weight Search — 4 Models (+ LR)
4D weight space shown as parallel coordinates plot.

In [10]:
mc4 = []
for _ in range(N_SAMPLES):
    a, b, c, d = random.random(), random.random(), random.random(), random.random()
    total = a + b + c + d
    a /= total; b /= total; c /= total; d /= total
    score = roc_auc_score(y, a*oof_cat + b*oof_lgb + c*oof_xgb + d*oof_lr)
    mc4.append([a, b, c, d, score])

mc4_df = pd.DataFrame(mc4, columns=["catboost", "lgbm", "xgb", "lr", "score"])
best4  = mc4_df.loc[mc4_df["score"].idxmax()]

print(f"Best MC weights (4-model):")
print(f"  catboost={best4['catboost']:.3f}  lgbm={best4['lgbm']:.3f}  "
      f"xgb={best4['xgb']:.3f}  lr={best4['lr']:.3f}")
print(f"  OOF AUC = {best4['score']:.5f}")
print(f"  vs simple avg (4) = {roc_auc_score(y, avg4):.5f}")

# Parallel coordinates — show top 20% by score
top20 = mc4_df[mc4_df["score"] >= mc4_df["score"].quantile(0.80)]
fig = px.parallel_coordinates(
    top20,
    dimensions=["catboost", "lgbm", "xgb", "lr", "score"],
    color="score",
    color_continuous_scale="RdYlGn",
    title="Monte Carlo: top 20% weight combinations (4-model)"
)
fig.show()

Best MC weights (4-model):
  catboost=0.553  lgbm=0.210  xgb=0.236  lr=0.001
  OOF AUC = 0.95538
  vs simple avg (4) = 0.95517


## SLSQP Optimization — 4 Models

In [11]:
oofs4  = [oof_cat, oof_lgb, oof_xgb, oof_lr]
names4 = ["catboost", "lgbm", "xgb", "lr"]

result4 = minimize(
    neg_auc, x0=[0.25]*4, args=(oofs4,),
    method="SLSQP",
    bounds=[(0, 1)] * 4,
    constraints={"type": "eq", "fun": lambda w: w.sum() - 1}
)
opt4_weights = result4.x
opt4_auc     = -result4.fun

print("SLSQP optimal weights (4-model):")
for n, w in zip(names4, opt4_weights):
    print(f"  {n:<12} {w:.4f}")
print(f"  OOF AUC = {opt4_auc:.5f}")

SLSQP optimal weights (4-model):
  catboost     0.2500
  lgbm         0.2500
  xgb          0.2500
  lr           0.2500
  OOF AUC = 0.95517


## Summary & Decision
Compare all approaches, decide which to submit.

In [12]:
results = {
    "simple_avg_3":  roc_auc_score(y, avg3),
    "simple_avg_4":  roc_auc_score(y, avg4),
    "mc_best_3":     best3["score"],
    "mc_best_4":     best4["score"],
    "slsqp_3":       opt3_auc,
    "slsqp_4":       opt4_auc,
}
print("OOF AUC comparison:")
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {k:<20} {v:.5f}")

best_combo = max(results, key=results.get)
print(f"\nBest: {best_combo} ({results[best_combo]:.5f})")

OOF AUC comparison:
  mc_best_3            0.95538
  mc_best_4            0.95538
  simple_avg_3         0.95537
  slsqp_3              0.95537
  simple_avg_4         0.95517
  slsqp_4              0.95517

Best: mc_best_3 (0.95538)


## Submit Best Ensemble

In [13]:
def submit_ensemble(test_preds, weights, names, label, oof_auc):
    pred = sum(w * p for w, p in zip(weights, test_preds))
    sub = ss.copy()
    sub["Heart Disease"] = pred
    fname = f"submissions/{label}.csv"
    desc  = f"{label} | cv_auc={oof_auc:.4f}"
    sub.to_csv(fname, index=False)
    r = subprocess.run(
        ["kaggle", "competitions", "submit", "-c", "playground-series-s6e2",
         "-f", fname, "-m", desc],
        capture_output=True, text=True
    )
    status = "submitted" if "successfully" in r.stdout.lower() else r.stdout.strip()[:80]
    print(f"{label}: {status}")
    print(f"  weights: {dict(zip(names, [round(w,3) for w in weights]))}")
    print(f"  desc: {desc}")

# Submit 3-model SLSQP ensemble
submit_ensemble(
    [test_cat, test_lgb, test_xgb], opt3_weights, names3,
    "ensemble_3gbt_slsqp", opt3_auc
)

# Submit 4-model if it's clearly better (> 0.0002 gap)
if opt4_auc - opt3_auc > 0.0002:
    print()
    submit_ensemble(
        [test_cat, test_lgb, test_xgb, test_lr], opt4_weights, names4,
        "ensemble_4model_slsqp", opt4_auc
    )
else:
    print(f"\n4-model gain ({opt4_auc - opt3_auc:+.5f}) too small — skipping to save slots")

ensemble_3gbt_slsqp: submitted
  weights: {'catboost': 0.333, 'lgbm': 0.333, 'xgb': 0.333}
  desc: ensemble_3gbt_slsqp | cv_auc=0.9554

4-model gain (-0.00020) too small — skipping to save slots
